### Question : Regional Purchase Activity (Join + Filter)

Scenario: You have two tables from a customer events pipeline — customers.csv (a dimension table) and events.csv (a raw events log). As often happens, the events log contains an event for a customer that doesn't exist in the customer dimension (a late-arriving or bad record) — you should not include such orphaned events in the analysis.

**Problem:**

- Read both files with explicit schemas.
- Join events to customers on customer_id — only keep events where a matching customer exists.
- Filter to event_type = 'PURCHASE' occurring on 2024-02-01 (date only, ignore time).
- Count the number of distinct customers who made a purchase on that date, grouped by region.
- Sort by count descending, then by region ascending.

**customers.csv Schema**

| Column | Type |
| :--- | :--- |
| **customer_id** | string |
| **customer_name** | string |
| **signup_date** | date |
| **region** | string |

**events.csv Schema**

| Column | Type |
| :--- | :--- |
| **event_id** | string |
| **customer_id** | string |
| **event_type** | string |
| **event_time** | timestamp |

**Expected Output**

| region | distinct_purchasers |
| :--- | :--- |
| East | 1 |
| North | 1 |
| West | 1 |

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import *

In [0]:
schema_customers = StructType(
    [
        StructField("customer_id", StringType()),
        StructField("customer_name", StringType()),
        StructField("signup_date", DateType()),
        StructField("region", StringType())
    ]
)

schema_events = StructType(
    [
        StructField("event_id", StringType()),
        StructField("customer_id", StringType()),
        StructField("event_type", StringType()),
        StructField("event_time", TimestampType())
    ]
)

In [0]:
df_customers = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema_customers)
    .load("/Workspace/Users/jeevan.azureacc2@gmail.com/spark-practice/data/customers.csv")
)

df_events = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema_events)
    .load("/Workspace/Users/jeevan.azureacc2@gmail.com/spark-practice/data/events.csv")
)

In [0]:
df_events = df_events.filter(
    (to_date(col("event_time")) == "2024-02-01") & (col("event_type") == "PURCHASE")
)

df_cust_by_region = (
    df_customers.join(df_events, "customer_id", "inner")
    .groupBy(col("region"))
    .agg(
        countDistinct(col("customer_id")).alias(
            "distinct_purchasers"
        )
    )
)
df_cust_by_region.sort(col("distinct_purchasers").desc(), col("region").asc()).show()

### Scenario: 
You're given dau_rolling_avg.csv, a daily active users feed for a product. Leadership wants a smoothed trend line instead of the noisy daily numbers.

**Problem:**

- Read the file with an explicit schema.
- Compute a 3-day trailing rolling average of active_users for each date (current day + previous 2 days). For the first 2 days, where fewer than 3 days of history exist, average over whatever days are available.
- Round the result to 2 decimal places.
- Order the output by date ascending.

**Schema**

| Column | Type |
| :--- | :--- |
| **date** | date |
| **active_users** | int |

**Expected Output**

| date | active_users | rolling_avg_3d |
| :--- | :--- | :--- |
| 2024-03-01 | 120 | 120.00 |
| 2024-03-02 | 135 | 127.50 |
| 2024-03-03 | 150 | 135.00 |
| 2024-03-04 | 110 | 131.67 |
| 2024-03-05 | 140 | 133.33 |
| 2024-03-06 | 160 | 136.67 |
| 2024-03-07 | 155 | 151.67 |
| 2024-03-08 | 130 | 148.33 |
| 2024-03-09 | 145 | 143.33 |
| 2024-03-10 | 170 | 148.33 |

In [0]:
from pyspark.sql.window import *

In [0]:
schema = StructType(
    [
        StructField("date", DateType()),
        StructField("active_users", IntegerType())
    ]
)

df = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema)
    .load("/Workspace/Users/jeevan.azureacc2@gmail.com/spark-practice/data/dau_rolling_avg.csv")
)

window_logic = Window.orderBy(col("date")).rowsBetween(-2, 0)

df = df.withColumn(
    "rolling_avg_3d",
    round(avg(col("active_users")).over(window_logic),2)
)

df.show()

%md
### Scenario:
You're given `app_error_logs.txt`, a raw semi-structured application log file (not CSV — space-delimited free text). The on-call team wants a quick breakdown of which services are throwing the most errors.

**Problem:**

- Read the file as plain text (one row per line, single string column).
- Using string/regex functions, parse out `log_date`, `log_time`, `log_level`, and `service` from each line. Each line follows the pattern: `yyyy-MM-dd HH:mm:ss LEVEL [ServiceName] user_id=... message=...`
- Filter to only `log_level = 'ERROR'`.
- Count the number of errors per `service`.
- Order the output by error count descending.

**Expected Output**

| service | error_count |
| :--- | :--- |
| PaymentService | 3 |
| AuthService | 2 |
| InventoryService | 1 |

In [0]:
df_api_logs = (
    spark.read.format("text")
    .load("/Workspace/Users/jeevan.azureacc2@gmail.com/spark-practice/data/app_error_logs.txt")
)

df_api_logs = df_api_logs.withColumns(
    {
        "log_date": regexp_extract(col("value"), r"(\d{4}-\d{2}-\d{2})", 1),
        "log_time": regexp_extract(col("value"), r"(\d{2}:\d{2}:\d{2})", 1),
        "log_level": regexp_extract(col("value"), r"\d{2}:\d{2}:\d{2} (\w+)", 1),
        "service": regexp_extract(col("value"), r"\[(.*?)\]", 1)
    }
)

df_errors_summary = (
    df_api_logs.groupBy(col("service"))
    .agg(count("value").alias("error_count"))
)

df_errors_summary.orderBy(col("error_count").desc()).show()

### Scenario:
You're maintaining a `customer_dim` Delta table (a slowly changing dimension, Type 1 — overwrite on change, no history kept). Today's incoming file `customer_dim_updates.csv` contains a mix of updates to existing customers, brand-new customers, and one bad record with a missing `customer_id` that must not pollute the dimension table.

**Problem:**

- Load `customer_dim_seed.csv` and write it as a managed Delta table named `customer_dim` (this represents "yesterday's" state of the table — do this once as setup).
- Read `customer_dim_updates.csv` as today's incoming batch.
- Split the incoming batch: rows with a null `customer_id` are bad records — write them to a separate `customer_dim_quarantine` Delta table instead of processing them further. Print the quarantined row count.
- For the remaining valid rows, perform a **merge (upsert)** into `customer_dim`:
  - If `customer_id` already exists in the table, update `customer_name`, `region`, and `updated_at`.
  - If `customer_id` does not exist, insert it as a new row.
- Display the final `customer_dim` table, sorted by `customer_id` ascending.

**Seed Schema (`customer_dim_seed.csv`)**

| Column | Type |
| :--- | :--- |
| **customer_id** | string |
| **customer_name** | string |
| **region** | string |
| **updated_at** | date |

**Updates Schema (`customer_dim_updates.csv`)** — same columns as seed.

**Expected Output — quarantine count**

| quarantined_count |
| :--- |
| 1 |

**Expected Output — final `customer_dim` table**

| customer_id | customer_name | region | updated_at |
| :--- | :--- | :--- | :--- |
| C001 | Alice Johnson | West | 2024-01-01 |
| C002 | Bob Singh | North | 2024-02-01 |
| C003 | Carol Mehta | East | 2024-02-01 |
| C004 | David Kim | South | 2024-02-01 |

In [0]:
%sql
create table if not exists pyspark_practice.default.customer_dim
(
    customer_id varchar(50),
    customer_name varchar(50),
    region varchar(50),
    updated_at Date
)
using delta;

In [0]:
schema = StructType(
    [
        StructField("customer_id", StringType()),
        StructField("customer_name", StringType()),
        StructField("region", StringType()),
        StructField("updated_at", DateType())
    ]
)

df_cust_seed = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema)
    .load("/Workspace/Users/jeevan.azureacc2@gmail.com/spark-practice/data/customer_dim_seed.csv")
)

df_cust_seed.write.format("delta").mode("overwrite").saveAsTable("pyspark_practice.default.customer_dim")

In [0]:
df_cust_update = (
    spark.read.format("csv")
    .option("header", True)
    .schema(schema)
    .load("/Workspace/Users/jeevan.azureacc2@gmail.com/spark-practice/data/customer_dim_updates.csv")
)

df_cust_quarantine = df_cust_update.filter(col("customer_id").isNull())
print(df_cust_quarantine.count())
df_cust_quarantine.write.mode("overwrite").format("delta").saveAsTable("pyspark_practice.default.customer_dim_quarantine")

df_cust_update= df_cust_update.filter(col("customer_id").isNotNull())

df_cust_table = DeltaTable.forName(spark, "pyspark_practice.default.customer_dim")

(
    df_cust_table.alias("t")
    .merge(df_cust_update.alias("s"), "t.customer_id = s.customer_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)


In [0]:
%sql
select * from pyspark_practice.default.customer_dim;
select * from pyspark_practice.default.customer_dim_quarantine;